# Training-free guidance (TFG) on Tahoe cell states

Comparison method 2 of 3 for the cell-state modality. The condition enters only through the
sampling trajectory: the frozen diffusion model runs with the drug id set to null, and at every
step the sample is pushed along the gradient of a fixed classifier score. Nothing is trained in
this notebook.

- Paper: Ye et al., *TFG: Unified Training-Free Guidance for Diffusion Models*, NeurIPS 2024,
  [arXiv:2409.15761](https://arxiv.org/abs/2409.15761).
- Official code: [YWolfeee/Training-Free-Guidance](https://github.com/YWolfeee/Training-Free-Guidance),
  `methods/tfg.py`. Our version is `src/guidance/baselines/tfg.py`; its header lists what was kept
  and what was changed.
- Diffusion model: the (drug, cell line) conditional MLP shared by all cell notebooks, given the
  pair's cell line and a null drug.
- Classifier that guides: one Gaussian per drug (linear discriminant analysis) fitted on the
  training cells. No pretrained cell-state classifier exists and this project trains none.
- Scorer: a separate k-nearest-neighbour vote over real training cells. The function that guides
  is never the one that scores, otherwise "accuracy" would only measure how well guidance games
  its own objective.

How to run: top to bottom on a GPU. Run `cfg_cells.ipynb` first on the same machine; its
`checkpoints/diffusion_cfg/best.pt` is loaded here (or the path in `CELL_BASE_CKPT`). A `base.pt`
written by any cell notebook is also accepted. If neither exists, the base is trained here with
the same recipe.

## What TFG does, and how this notebook maps onto it

Notation: $x_k$ is the noisy cell state at step $k$ ($K = 1000$ training steps; the network
receives $t = k/K$), $k'$ the next of the 250 sampling steps, $\bar\alpha_k$ the cosine schedule,
$\varepsilon_\theta$ the frozen noise-prediction MLP over 50 principal components, $\ell$ the pair's
cell line, $\varnothing$ the null drug, $y$ the target drug. The classifier is
$\log p_r(y \mid x) = \log\operatorname{softmax}_d\!\big(-\tfrac12 (x-\mu_d)^\top\Sigma^{-1}(x-\mu_d) + \log\pi_d\big)_y$
with per-drug means $\mu_d$, one shared covariance $\Sigma$ and class frequencies $\pi_d$.

At every step TFG estimates the clean state, scores it, and moves along the gradient. There are
two terms: a *variance* term, the gradient with respect to the noisy state taken through the
denoiser, and a *mean* term, gradient steps on the clean estimate itself. Both act on a smoothed
objective and both have a schedule over the trajectory. Per step $k \to k'$ (Algorithm 1 of the
paper):

$$\hat x_0 = \operatorname{clip}\!\Big(\frac{x_k - \sqrt{1-\bar\alpha_k}\;\varepsilon_\theta(x_k, k, \varnothing, \ell)}{\sqrt{\bar\alpha_k}}\Big),
\qquad
\log \tilde f(x) = \log \frac{1}{n_\varepsilon}\sum_{i=1}^{n_\varepsilon} \exp f\big(x + \sigma^{\mathrm s}_k\,\delta_i\big),\quad f(x) = \log p_r(y \mid x)$$

$$\Delta_k = \rho_k\, \nabla_{x_k} \log \tilde f\big(\hat x_0(x_k)\big)
\qquad\text{(variance guidance; the gradient goes through } \varepsilon_\theta)$$

$$\hat x_0^{(j+1)} = \hat x_0^{(j)} + \mu_k\, \nabla \log \tilde f\big(\hat x_0^{(j)}\big),\ \ j = 0,\dots,N_{\text{iter}}-1,
\qquad \Delta_0 = \hat x_0^{(N_{\text{iter}})} - \hat x_0
\qquad\text{(mean guidance)}$$

$$x_{k'} = \underbrace{\sqrt{\bar\alpha_{k'}}\,\hat x_0 + \sqrt{1-\bar\alpha_{k'}-\sigma_k^2}\;\hat\varepsilon + \sigma_k z}_{\text{ancestral step, } \hat\varepsilon \text{ recomputed from the clipped } \hat x_0}
\;+\; \frac{\Delta_k}{\sqrt{\bar\alpha_k / \bar\alpha_{k'}}} \;+\; \sqrt{\bar\alpha_{k'}}\;\Delta_0$$

The schedules are theirs: $\rho_k = \rho\, a_k / \bar a$ and $\mu_k = \mu\, a_k / \bar a$ with
$a_k = \bar\alpha_k / \bar\alpha_{k'}$ and $\bar a$ its mean over the trajectory, so the average
strength equals the $\rho$ and $\mu$ you set; the smoothing scale is
$\sigma^{\mathrm s}_k = \sigma^{\mathrm s}\sqrt{1-\bar\alpha_k}$ (the paper writes $\sigma_t$; it is
renamed here so it is not confused with the ancestral noise scale $\sigma_k$).

The clip. Their code clamps $\hat x_0$ to $[-1, 1]$, the pixel range. Cell states have no fixed
range, so here each principal component is clamped to its training range, which an in-range
estimate never touches. The clamp matters: at the first sampling step $\bar\alpha_k$ is about
$2\times10^{-9}$, so $\hat x_0$ divides the denoiser's residual by about $5\times10^{-5}$ and
multiplies any error in $\varepsilon_\theta$, and in its Jacobian (which $\Delta_k$ goes through),
by about 20,000. The first run of this notebook had no clamp and its samples left the data range
by 300 standard deviations. The same clamp is applied to every row so rows stay comparable, and
the fraction of coordinates that hit it is reported. `CFG_CLIP_X0=0` turns it off.

| Paper / official code | Here |
|---|---|
| Alg. 1: clean estimate $\hat x_0$, clamped to $[-1, 1]$ | `TFG._predict_x0`, clamped to the per-PC training range (`X0_BOUNDS`) |
| Alg. 1: variance guidance $\Delta_k$ through the denoiser | the `rho` block of `TFG.sample` |
| Alg. 1: mean guidance, $N_{\text{iter}}$ steps on $\hat x_0$ | the `iter_steps` loop of `TFG.sample` |
| Alg. 1: the update to $x_{k'}$; `_predict_x_prev_from_zero` recomputes $\hat\varepsilon$ from the clipped $\hat x_0$ | `TFG.sample`; `ddim_sample` below uses the same convention, checked at $\rho = \mu = 0$ |
| smoothed objective $\log\tilde f$ (`tilde_get_guidance`) | `TFG._log_f_tilde` |
| schedules `get_rho`, `get_mu`, `get_std` | `TFG._strength`, `TFG._std` |
| `ImageLabelGuidance`: a classifier's $\log p(y \mid \hat x_0)$ | `make_log_p` on the Gaussian classifier (`guide_logits`) |
| `scripts/cifar10_label.sh`: $\rho = 1$, $\mu = 0.25$, $\sigma^{\mathrm s} = 10^{-3}$, $n_\varepsilon = 1$, $N_{\text{iter}} = 4$, one recurrence, 50 steps | the `Cfg` defaults, with 250 steps (the cell sampler's) |
| an unconditional diffusion model | `eps_line`: the conditional model with the line given and the drug null |
| not in TFG | the second (kNN) scorer, the wrong-drug row, the label-conditioned reference row, paired generators, the strength sweep |

Differences from the paper. Their sampler is stochastic DDIM with $\eta = 1$, which is ancestral
DDPM, and so is ours (250 steps, as every cell notebook; the image notebooks use $\eta = 0$).
Their model is an unconditional CIFAR-10 DDPM; ours is a conditional MLP with the line given and
the drug null. Their objective is an image classifier; ours is a Gaussian classifier on cell
states. Their clamp is $[-1, 1]$; ours is the per-PC training range. They tuned hyperparameters
per task; we use their CIFAR-10 values unchanged, so these settings are not tuned for cells. The
header of `tfg.py` lists eight deviations.

In [ ]:
# Colab bootstrap. Run once.
import importlib.util, subprocess, sys, os

for pkg in ("datasets",):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

if importlib.util.find_spec("guidance") is None:
    # Inside a checkout (Jupyter on a server, cwd somewhere under the repo): walk up to its src/.
    from pathlib import Path as _P
    _src = next((p / "src" for p in [_P.cwd(), *_P.cwd().parents] if (p / "src" / "guidance").is_dir()), None)
    if _src is None:                                   # Colab: clone the repo next to the notebook
        if not os.path.exists("discrete-generative-models-proj-1"):
            subprocess.run(["git", "clone", "-q",
                            "https://github.com/imperorrp/discrete-generative-models-proj-1.git"],
                           check=True)
        _src = "discrete-generative-models-proj-1/src"
    sys.path.insert(0, str(_src))
    print("guidance package from:", _src)

import torch
from guidance.baselines.tfg import TFG, TFGConfig

def _git_sha():
    for d in ("discrete-generative-models-proj-1", "."):
        try:
            return subprocess.check_output(["git", "-C", d, "rev-parse", "--short", "HEAD"],
                                           text=True, stderr=subprocess.DEVNULL).strip()
        except Exception:
            pass
    return "unknown"

GIT_SHA = _git_sha()
print("code version:", GIT_SHA)
print("device:", "cuda" if torch.cuda.is_available()
      else "CPU  <-- Runtime > Change runtime type > GPU")

## Config

All settings in one place. Any field can be set from the environment as `CFG_<NAME>`, for
example `CFG_LIMIT=5000 CFG_BASE_EPOCHS=1` for a smoke run, so a headless run never needs a cell
edited.

In [ ]:
import math, time, json
from dataclasses import dataclass, asdict, fields
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class Cfg:
    seed: int = 42                 # training / sampling randomness, as cfg_cells
    split_seed: int = 42           # shuffling only; the Tahoe split is fixed by the data files
    batch_size: int = 256
    base_epochs: int = 50
    lr_base: float = 2e-4
    lr_schedule: str = "constant"     # cosine anneals to zero; "constant" is the cell base's recipe
    amp: bool = True               # fp16 autocast on CUDA; the cell base's recipe
    K: int = 1000                  # diffusion training steps
    sample_steps: int = 250        # sampling steps, as the cell base's own notebook
    reuse_base: bool = True        # load base.pt if present instead of retraining
    limit: int = 0                 # >0 truncates the training set, for smoke tests
    samples_per_condition: int = 400
    n_eval_conditions: int = 6     # held-out pairs evaluated: the largest pair of each of this many drugs
    sweep: tuple = (0.0, 0.5, 1.0, 2.0, 4.0)
    shrinkage: float = 1e-2        # added to the diagonal of the shared covariance
    condition: str = 'pair'        # pair (drug + cell line) | drug; see guidance.data.cells
    dose: float = -1.0             # -1 keeps every dose; 0.05, 0.5 or 5 keeps that dose (uM) plus controls
    clip_x0: float = -1.0          # clamp of the predicted clean state: -1 = per-PC training range (cells) or [-1, 1] (images); 0 = off; c > 0 = [-c, c]
    # TFG -- defaults are the official CIFAR-10 label-guidance script
    rho: float = 1.0
    mu: float = 0.25
    sigma: float = 0.001
    eps_bsz: int = 1
    iter_steps: int = 4
    recur_steps: int = 1
    eta: float = 1.0               # ancestral DDPM, as the cell base's own notebook and TFG's own CIFAR-10 script

cfg = Cfg()

# Any field can be overridden from the environment, e.g. CFG_BASE_EPOCHS=1 CFG_LIMIT=5000, so
# headless runs (nbconvert, a server, a different GPU) never need a cell edited.
for _f in fields(Cfg):
    _v = os.environ.get(f"CFG_{_f.name.upper()}")
    if _v is not None:
        _cur = getattr(cfg, _f.name)
        setattr(cfg, _f.name, tuple(float(t) for t in _v.split(",")) if isinstance(_cur, tuple)
                else type(_cur)(_v) if not isinstance(_cur, bool) else _v.lower() in ("1", "true", "yes"))
        print(f"cfg.{_f.name} <- {getattr(cfg, _f.name)!r} (from CFG_{_f.name.upper()})")
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
print(DEV, asdict(cfg))

## Data

Tahoe-100M subset: 25 cell lines x 100 drugs at three doses, plus DMSO controls; 991,899 cells.
Each cell is 2000 highly variable genes reduced to 50 principal components. Normalisation, gene
selection, scaling and PCA were fitted on the training split only
(`notebooks/tahoe/tahoe_colab_downloader_with_pca.ipynb`). About 20% of (cell line, drug) pairs
are held out: every validation cell comes from a pair never seen in training, while its line and
its drug each appear in other training pairs, so validation loss already measures generalisation
to unseen combinations. `x` is the 50-vector, scaled so its overall variance is 1.

The condition id is the (cell line, drug) pair, 2525 ids. The backbone turns it into a drug
embedding plus a cell-line embedding, so an unseen pair is a seen drug plus a seen line. Each
pair's text is the drug's mechanism of action and targets plus the cell line's name and organ,
for example "Binimetinib, mek inhibitor targeting MAP2K1, MAP2K2, in HepG2/C3A liver cells",
written from the public metadata before any result was seen.

The loader looks for the unpacked `25lines_100drugs_pca_data_v2.zip` next to the notebook, under
`data/` in the repo, or on Colab and Drive; set `TAHOE_PCA_DIR` if it is somewhere else.

In [ ]:
from guidance.data.cells import make_loaders

train_loader, val_loader, meta = make_loaders(
    batch_size=cfg.batch_size, seed=cfg.split_seed, condition=cfg.condition,
    dose=None if cfg.dose < 0 else cfg.dose, limit=cfg.limit)

xb, cb = next(iter(train_loader))
print("x", tuple(xb.shape), f"rms {float(xb.pow(2).mean().sqrt()):.2f}", "| cond", tuple(cb.shape))
print(f"train cells {meta['n_train']:,} | val cells {meta['n_val']:,} | "
      f"conditions {meta['n_conditions']} ({meta['condition']}) | split: {meta['split']}")
print("example text:", meta["cond_text"][1])

## The objective: a Gaussian classifier per drug

No pretrained classifier exists for these labels and this project trains none, so the classifier
is fitted by moments on the training cells: one mean per drug, one shared covariance of the
within-drug residuals plus a small diagonal shrinkage, class frequencies as priors. That is
linear discriminant analysis. Its log-posterior is differentiable in $x$, which is all TFG needs.
The target is the drug of each evaluation pair.

The scorer is a different function on the same labels: a k-nearest-neighbour vote over real
training cells with their true drug. This cell prints both scorers' accuracy on real validation
cells. Expect low numbers: drug effects are small next to cell-line differences, and the
validation cells are unseen line-drug pairs. These numbers bound what guidance can reach.

In [ ]:
from guidance.data.cells import condition_texts

# Drug of every condition id, from the loader's table (the backbone cell builds its own copy later).
COND_DRUG = meta["cond_ids_table"][:, 0].to(DEV)
X_TRAIN = train_loader.dataset.x.to(DEV); Y_TRAIN = COND_DRUG[train_loader.dataset.cond.to(DEV)]   # drug of each cell
X_VAL = val_loader.dataset.x.to(DEV);     Y_VAL = COND_DRUG[val_loader.dataset.cond.to(DEV)]
N_DRUGS, D = meta["n_drugs"], X_TRAIN.shape[1]
_drug_text, _ = condition_texts()
DRUG_TEXT = [_drug_text[n] for n in meta["drug_names"]]

# Fitted by moments on the training cells. Nothing is trained.
COUNTS = torch.bincount(Y_TRAIN, minlength=N_DRUGS).float()
MU = torch.zeros(N_DRUGS, D, device=DEV).index_add_(0, Y_TRAIN, X_TRAIN) / COUNTS.clamp(min=1)[:, None]
RES = X_TRAIN - MU[Y_TRAIN]
SIGMA = RES.T @ RES / len(RES) + cfg.shrinkage * torch.eye(D, device=DEV)
PREC = torch.linalg.inv(SIGMA)
LOG_PRIOR = torch.log(COUNTS.clamp(min=1) / COUNTS.sum())


def guide_logits(x):
    '''Unnormalised log p(y | x) of the Gaussian classifier, for every drug. Differentiable in x.'''
    d = x[:, None, :] - MU[None]                              # (B, N_DRUGS, D)
    return -0.5 * torch.einsum("bkd,de,bke->bk", d, PREC, d) + LOG_PRIOR


def lda_label(x):
    return guide_logits(x).argmax(1)


# Independent scorer: the shared k-nearest-neighbour drug scorer over real training cells.
# Same labels, different decision function, never seen by the sampler.
from guidance.eval.cell_metrics import all_metrics, KNNConditionScorer
SCORER = KNNConditionScorer(X_TRAIN, Y_TRAIN, k=25, seed=cfg.seed, device=DEV)
knn_label = SCORER.predict
KNN_K, KNN_REF = SCORER.k, SCORER.x


# Ceilings: how often each scorer recognises the drug of a REAL validation cell. These bound
# what guidance toward a drug can reach on this data, and they are on unseen line-drug pairs.
n_check = min(20_000, len(X_VAL))
LDA_REAL_ACC = (lda_label(X_VAL[:n_check]) == Y_VAL[:n_check]).float().mean().item()
KNN_REAL_ACC = (knn_label(X_VAL[:n_check]) == Y_VAL[:n_check]).float().mean().item()
print(f"Gaussian classifier: {N_DRUGS} drugs, shared {D}x{D} covariance, fitted on {len(X_TRAIN):,} training cells")
print(f"accuracy on {n_check:,} real validation cells: Gaussian classifier {LDA_REAL_ACC:.1%} | "
      f"{KNN_K}-NN scorer {KNN_REAL_ACC:.1%} | chance {1 / N_DRUGS:.1%}")
if KNN_REAL_ACC < 3 / N_DRUGS:
    print("the scorer barely recognises real drug cells; guidance accuracy below will be near chance too. "
          "Report it, and read the per-drug MMD instead.")

## Backbone

The MLP from the teammate's `Tahoe_PCA_Diffusion_CFG.ipynb`, kept as written: the drug and the
cell line are embedded separately, summed with the time embedding, and that vector enters every
residual block. `CellDiffusionMLP` wraps it so the notebooks can pass one pair id and `t` in
[0, 1]: `COND_TABLE` maps the pair id to (drug id, line id), with an extra last row for the null
condition (both ids null), and `t` is scaled back to the integer step the time embedding was
written for.

In [ ]:
def timestep_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10_000.0) * torch.arange(half, device=t.device) / half)
    args = t.float()[:, None] * freqs[None, :]
    return torch.cat([args.cos(), args.sin()], dim=-1)

# ---- The model of notebooks/tahoe/Tahoe_PCA_Diffusion_CFG.ipynb, a teammate's classifier-free-
# guidance base for these cells, kept verbatim: drug and cell line are embedded separately and
# summed with the time embedding into one vector that enters every residual block.
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        half = dim // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half) / max(half - 1, 1))
        self.register_buffer("freqs", freqs, persistent=False)

    def forward(self, t):
        args = t.float()[:, None] * self.freqs[None, :].to(t.device)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class ResidualBlock(nn.Module):
    def __init__(self, dim, cond_dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1 = nn.Linear(dim + cond_dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.act = nn.SiLU()

    def forward(self, x, cond):
        h = torch.cat([self.norm(x), cond], dim=-1)
        h = self.fc2(self.act(self.fc1(h)))
        return x + h


class DiffusionMLP(nn.Module):
    def __init__(self, data_dim, n_drugs, n_lines, time_dim=128, cond_dim=128, hidden_dim=256, n_blocks=4):
        super().__init__()
        self.data_dim = data_dim
        self.time_embed = SinusoidalTimeEmbedding(time_dim)
        self.time_mlp = nn.Sequential(nn.Linear(time_dim, cond_dim), nn.SiLU(), nn.Linear(cond_dim, cond_dim))
        self.drug_embed = nn.Embedding(n_drugs + 1, cond_dim)   # last row is the null/unconditional id
        self.line_embed = nn.Embedding(n_lines + 1, cond_dim)   # last row is the null/unconditional id
        self.input_proj = nn.Linear(data_dim, hidden_dim)
        self.blocks = nn.ModuleList([ResidualBlock(hidden_dim, cond_dim) for _ in range(n_blocks)])
        self.output_proj = nn.Linear(hidden_dim, data_dim)

    def forward(self, x, t, drug_id, line_id):
        cond = self.time_mlp(self.time_embed(t)) + self.drug_embed(drug_id) + self.line_embed(line_id)
        h = self.input_proj(x)
        for block in self.blocks:
            h = block(h, cond)
        return self.output_proj(h)


# ---- Our wrapper. The notebooks pass one cond_id and t in [0, 1]. The loader's table turns the
# id into (drug_id, line_id); the extra last row is the null condition, both ids null, which is
# their joint dropout. t is scaled back to the integer step their time embedding was written for.
COND_TABLE = torch.cat([meta["cond_ids_table"],
                        torch.tensor([[meta["n_drugs"], meta["n_lines"]]])]).to(DEV)
assert int(COND_TABLE.min()) >= 0, "this backbone needs condition = 'pair' or 'drug'"


class CellDiffusionMLP(DiffusionMLP):
    def __init__(self):
        super().__init__(meta["shape"][0], n_drugs=meta["n_drugs"], n_lines=meta["n_lines"])
        self.null = meta["n_conditions"]

    def forward(self, x, t, cond_ids=None):
        if cond_ids is None:
            cond_ids = torch.full((x.shape[0],), self.null, device=x.device, dtype=torch.long)
        ids = COND_TABLE[cond_ids]
        return super().forward(x, t * cfg.K, ids[:, 0], ids[:, 1])

    def forward_ids(self, x, t, drug_id, line_id):
        '''Direct (drug, line) ids, for partially conditioned branches such as line known, drug null.'''
        return super().forward(x, t * cfg.K, drug_id, line_id)


def make_backbone():
    return CellDiffusionMLP()

# Only the trunk is adapted by TC-LoRA: the input projection and the residual blocks' two
# linear layers each. Time MLP, the two embeddings and the output projection stay untouched.
SKIP_NAMES = ("output_proj", "time_mlp", "drug_embed", "line_embed")

torch.manual_seed(cfg.seed)      # model init and training randomness follow `seed`
net = make_backbone().to(DEV)
print(f"backbone params: {sum(p.numel() for p in net.parameters()):,}")

## Diffusion: schedule, noising, sampler

The cosine schedule as diffusers builds it (`squaredcos_cap_v2`), which is the schedule the cell
base was trained with. `add_noise` draws a random step and a noise vector for training.
`ddim_sample` is the ancestral sampler (`eta = 1`, 250 steps) written out in plain PyTorch, the
same update as the diffusers `DDPMScheduler` step used in `cfg_cells`, plus a clamp on the
predicted clean state. The TFG sampler in `src/` reduces to exactly this function when guidance
is off, and the checks below assert it.

In [ ]:
def cosine_alpha_bar(K, s=0.008, device="cpu"):
    steps = torch.arange(K + 1, dtype=torch.float32, device=device) / K
    f = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
    return (f / f[0])[1:].clamp(1e-5, 0.9999)

def cosine_alpha_bar_diffusers(K, max_beta=0.999, device="cpu"):
    '''diffusers "squaredcos_cap_v2": the same cosine curve turned into betas clipped at
    max_beta, then a cumulative product. The schedule the cell base (cfg_cells) was trained with.'''
    steps = torch.arange(K + 1, dtype=torch.float64, device=device) / K
    f = torch.cos((steps + 0.008) / 1.008 * math.pi / 2) ** 2
    betas = (1 - f[1:] / f[:-1]).clamp(max=max_beta)
    return torch.cumprod(1 - betas, 0).float()

ABAR = cosine_alpha_bar(cfg.K, device=DEV) if len(meta["shape"]) == 3 else cosine_alpha_bar_diffusers(cfg.K, device=DEV)
NULL_ID = meta["n_conditions"]

# ---- Clamp on the predicted clean state x0_hat (TFG's clip_sample): [-1, 1] for images; for
# cells, each principal component's training range, which an in-range estimate never touches.
# Why: at the first sampling step abar is 2e-9, so x0_hat = (x - sqrt(1 - abar) eps) / sqrt(abar)
# multiplies any error in eps, and any error in its Jacobian that variance guidance goes through,
# by about 20,000. The first run of this notebook had no clamp and its samples left the data
# range by 300 standard deviations. The same bounds apply to every row so rows stay comparable,
# and the on-clamp fraction is reported. CFG_CLIP_X0=0 turns the clamp off.
def x0_bounds(c):
    '''(lo, hi) tensors that broadcast over a batch, or None for no clamp. See cfg.clip_x0.'''
    if c == 0:
        return None
    if c > 0:
        return torch.tensor(-float(c), device=DEV), torch.tensor(float(c), device=DEV)
    if len(meta["shape"]) == 3:
        return torch.tensor(-1.0, device=DEV), torch.tensor(1.0, device=DEV)
    _xtr = train_loader.dataset.x
    return _xtr.min(0).values.to(DEV), _xtr.max(0).values.to(DEV)

def clamp_x0(x0, bounds):
    return x0 if bounds is None else torch.maximum(torch.minimum(x0, bounds[1]), bounds[0])

def saturation(v, bounds=None):
    '''Fraction of coordinates sitting on the clamp; the objective's gradient is zero there.'''
    bounds = X0_BOUNDS if bounds is None else bounds
    if bounds is None:
        return float("nan")
    return ((v <= bounds[0] + 1e-6) | (v >= bounds[1] - 1e-6)).float().mean().item()

X0_BOUNDS = x0_bounds(cfg.clip_x0)
CLIP_X0 = X0_BOUNDS if X0_BOUNDS is not None else 0.0          # what TFGConfig receives
X0_CLAMP_DESC = ("off" if X0_BOUNDS is None else
                 "per-PC training range" if cfg.clip_x0 < 0 and len(meta["shape"]) == 1 else
                 f"[-{X0_BOUNDS[1].max().item():g}, {X0_BOUNDS[1].max().item():g}]")
print("x0 clamp:", X0_CLAMP_DESC,
      "" if X0_BOUNDS is None else f"(lo {X0_BOUNDS[0].min().item():.1f}..{X0_BOUNDS[0].max().item():.1f}, hi {X0_BOUNDS[1].min().item():.1f}..{X0_BOUNDS[1].max().item():.1f})")
print(f"abar at the first sampling step {ABAR[-1].item():.1e}: x0_hat amplifies the denoiser's residual by {1 / ABAR[-1].sqrt().item():,.0f}")


def add_noise(x0, gen=None):
    n = x0.shape[0]
    k = torch.randint(0, cfg.K, (n,), device=x0.device, generator=gen)
    eps = torch.randn(x0.shape, device=x0.device, generator=gen)
    ab = ABAR[k].view(n, *([1] * (x0.dim() - 1)))
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps, k, eps


def drop_labels(y, p_drop=0.1):
    keep = torch.rand(y.shape[0], device=y.device) >= p_drop
    return torch.where(keep, y, torch.full_like(y, NULL_ID))


@torch.no_grad()
def ddim_sample(eps_fn, n, shape, steps, clip_x0=None, x_init=None, eta=None, generator=None):
    '''The plain sampler, TFG's convention: clip x0_hat, then RECOMPUTE eps from the clipped
    x0_hat before stepping. eta = 0 is deterministic DDIM; eta = 1 (cells) is ancestral DDPM,
    with fresh noise at every step drawn from `generator` so paired rows stay paired. The TFG
    sampler in src/ reduces to exactly this at rho = mu = 0, and the cell below asserts it,
    at eta = 0 and at the notebook's own eta with identically seeded generators.
    clip_x0: None uses the notebook's X0_BOUNDS; 0 disables; c > 0 clamps to [-c, c].'''
    bounds = X0_BOUNDS if clip_x0 is None else x0_bounds(clip_x0)
    idx = torch.linspace(cfg.K - 1, 0, steps).long().to(DEV)
    eta = cfg.eta if eta is None else eta
    x = torch.randn(n, *shape, device=DEV, generator=generator) if x_init is None else x_init.clone()
    for i, k in enumerate(idx):
        t = (k.float() / cfg.K).repeat(n)
        eps = eps_fn(x, t)
        ab = ABAR[k]
        ab_prev = ABAR[idx[i + 1]] if i + 1 < len(idx) else torch.ones_like(ab)
        x0 = clamp_x0((x - (1 - ab).sqrt() * eps) / ab.sqrt(), bounds)
        eps = (x - ab.sqrt() * x0) / (1 - ab).sqrt()          # recompute from (the clipped) x0, as TFG does
        sigma = eta * ((1 - ab_prev) / (1 - ab) * (1 - ab / ab_prev)).sqrt()
        x = ab_prev.sqrt() * x0 + (1 - ab_prev - sigma ** 2).clamp(min=0).sqrt() * eps
        if eta > 0:
            x = x + sigma * torch.randn(x.shape, device=DEV, generator=generator)
    return x

## Base model: load the `cfg_cells` base, or `base.pt`, or train

In order of preference: a `base.pt` written by any cell notebook in this folder; the `cfg_cells`
checkpoint `checkpoints/diffusion_cfg/best.pt` (or the path in `CELL_BASE_CKPT`), which is the
base every cell notebook should share; otherwise the base is trained here with the same recipe.
A reused checkpoint is checked against this run's split, schedule and backbone first. The base's
per-epoch history travels with the checkpoint, so the training-curve figure shows the curve of
whichever run trained it, and the manifest records where the base came from.

Training and validation use different losses on purpose. Training drops the label 10% of the
time so the null branch exists (and so the null-label setting is in-distribution for the frozen
model). Validation does not: it scores the model on the same fixed noise and timesteps every
epoch, so the number is comparable across notebooks.

In [ ]:
VAL_SEED = cfg.seed + 999


def run_epochs(model, params, n_epochs, lr, tag, train_fn, val_fn, ckpt):
    # cfg.lr_schedule: "cosine" anneals to zero over the run (images); "constant" holds lr, the
    # cell base's recipe. cfg.amp: fp16 autocast with a grad scaler on CUDA, also the cell base's
    # recipe; the loss itself is computed in float32 either way.
    opt = torch.optim.AdamW(params, lr=lr)
    sched = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(n_epochs, 1)) if cfg.lr_schedule == "cosine"
             else torch.optim.lr_scheduler.LambdaLR(opt, lambda _: 1.0))
    use_amp = bool(cfg.amp) and DEV.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    hist, best, best_ep = [], float("inf"), 0
    for ep in range(1, n_epochs + 1):
        model.train(); tot = num = 0; t0 = time.time()
        for x, y in train_loader:
            x, y = x.to(DEV), y.to(DEV)
            with torch.autocast(device_type="cuda", enabled=use_amp):
                loss = train_fn(x, y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update()
            tot += loss.item() * x.shape[0]; num += x.shape[0]
        sched.step()
        val = validate(model, val_fn)
        is_best = val < best
        if is_best:
            best, best_ep = val, ep
            torch.save({"state": model.state_dict(), "epoch": ep, "val": val}, ckpt)
        hist.append({"epoch": ep, "train": tot / num, "val": val, "best": is_best,
                     "seconds": time.time() - t0})
        print(f"{tag} epoch {ep:3d}  train {tot/num:.4f}  val {val:.4f}"
              f"{'  *best' if is_best else ''}  ({time.time()-t0:.0f}s)")
    print(f"{tag}: best val {best:.4f} at epoch {best_ep}, saved to {ckpt}")
    return hist


@torch.no_grad()
def validate(model, val_fn):
    model.eval()
    gen = torch.Generator(device=DEV).manual_seed(VAL_SEED)
    tot = num = 0
    for x, y in val_loader:
        x, y = x.to(DEV), y.to(DEV)
        tot += val_fn(x, y, gen).item() * x.shape[0]; num += x.shape[0]
    return tot / num


def base_train(x, y):
    xk, k, eps = add_noise(x)
    return F.mse_loss(net(xk, k.float() / cfg.K, drop_labels(y)), eps)


def base_val(x, y, gen):
    xk, k, eps = add_noise(x, gen)
    return F.mse_loss(net(xk, k.float() / cfg.K, y), eps)


def hist_from_cfg_ckpt(ck):
    '''cfg_cells' checkpoint carries its per-epoch history. Recorded as it was (its own noise
    draws), without seconds, which are not this run's time.'''
    h = ck.get("history") or []
    return ([{"epoch": r["epoch"], "train": r["train_loss"], "val": r["val_loss"], "best": r["epoch"] == ck["epoch"]}
             for r in h]
            or [{"epoch": ck["epoch"], "train": float("nan"), "val": ck["val_loss"], "best": True}])


CKPT_META = {"split_seed": cfg.split_seed, "n_conditions": meta["n_conditions"],
             "shape": list(meta["shape"]), "K": cfg.K, "schedule": "cosine",
             "backbone_params": sum(p.numel() for p in net.parameters())}

if cfg.reuse_base and os.path.exists("base.pt"):
    ck = torch.load("base.pt", map_location=DEV, weights_only=False)
    saved = ck.get("meta")
    if saved is None:
        print("WARNING: base.pt carries no setup metadata (older run); split and schedule "
              "cannot be verified.")
    else:
        mismatch = {k: (saved.get(k), v) for k, v in CKPT_META.items() if saved.get(k) != v}
        assert not mismatch, f"base.pt was trained under a different setup: {mismatch}"
    net.load_state_dict(ck["state"])
    base_hist = ck.get("history") or [{"epoch": ck["epoch"], "train": float("nan"), "val": ck["val"], "best": True}]
    BASE_SOURCE = f"base.pt (from {ck.get('source', 'an earlier cell notebook run')})"
    print(f"loaded base.pt (epoch {ck['epoch']}, val {ck['val']:.4f}); skipping base training")
elif cfg.reuse_base and len(meta["shape"]) == 1 and (_cfg_ckpt := next(
        (p for p in [os.environ.get("CELL_BASE_CKPT"), "checkpoints/diffusion_cfg/best.pt",
                     "../checkpoints/diffusion_cfg/best.pt", "../../checkpoints/diffusion_cfg/best.pt"]
         if p and os.path.exists(p)), None)):
    # The base trained by cfg_cells.ipynb: same model class, so the weights load directly, and
    # every cell notebook sits on the same weights. Its validation loss was computed with that
    # notebook's noise draws, so it is reported as recorded, not recomputed.
    ck = torch.load(_cfg_ckpt, map_location=DEV, weights_only=False)
    assert ck["meta"]["n_params"] == sum(p.numel() for p in net.parameters()) and \
        ck["meta"]["num_train_timesteps"] == cfg.K, f"{_cfg_ckpt} does not match this backbone"
    net.load_state_dict(ck["model"])
    base_hist = hist_from_cfg_ckpt(ck)
    BASE_SOURCE = _cfg_ckpt
    torch.save({"state": net.state_dict(), "epoch": ck["epoch"], "val": ck["val_loss"], "meta": CKPT_META,
                "history": base_hist, "source": _cfg_ckpt}, "base.pt")
    print(f"loaded the cfg_cells base from {_cfg_ckpt} (epoch {ck['epoch']}, val {ck['val_loss']:.4f}); "
          f"saved as base.pt; skipping base training")
else:
    base_hist = run_epochs(net, list(net.parameters()), cfg.base_epochs, cfg.lr_base,
                           "base", base_train, base_val, "base.pt")
    BASE_SOURCE = "trained in this notebook"
    ck = torch.load("base.pt", map_location=DEV, weights_only=False)
    ck["meta"] = CKPT_META; ck["source"] = "tfg_cells"
    ck["history"] = [{k: v for k, v in h.items() if k != "seconds"} for h in base_hist]
    torch.save(ck, "base.pt")

# Whichever branch ran, sample from the BEST checkpoint on disk, not the last epoch left in
# `net`. Without this a fresh run and a reused-checkpoint run would use different weights
# while the manifest reported the same best validation loss for both.
ck = torch.load("base.pt", map_location=DEV, weights_only=False)
net.load_state_dict(ck["state"])
base_frozen = net.eval()
for p in base_frozen.parameters():
    p.requires_grad_(False)
print(f"sampling from base.pt epoch {ck['epoch']} (val {ck['val']:.4f}); base source: {BASE_SOURCE}")

## Set up TFG

TFG expects an unconditional denoiser. Ours is (drug, cell line) conditional with null ids, so
the drug id is null throughout and the classifier gradient is the only way the drug enters. The
cell line stays on (`eps_line`, defined with the evaluation pairs below): a drug classifier
cannot supply the line, and every cell notebook is evaluated per (line, drug) pair. `eps_uncond`,
with both ids null, is used only for the two checks in this cell.

The objective is `log p(y | x̂₀)` from the Gaussian classifier on the predicted clean state, the
role `ImageLabelGuidance` plays in their code.

Two checks run before anything else. With `rho = mu = 0` the TFG sampler must reproduce the plain
sampler exactly: at `eta = 0`, and at the notebook's `eta` with identically seeded generators
(the guided sampler draws exactly the noise the plain one draws). If either fails, stop.

In [ ]:
def eps_uncond(x, t):
    # Both ids null: the base's fully unconditional branch, used only for the checks below.
    return base_frozen(x, t, torch.full((x.shape[0],), NULL_ID, device=x.device, dtype=torch.long))


def make_log_p(y):
    '''log p(y | x0) under the Gaussian classifier, for a fixed batch of target drugs y.
    This is the role of ImageLabelGuidance: the quantity Algorithm 1 climbs at every step.'''
    def log_p(x0):
        logits = guide_logits(x0)
        yy = y.repeat(x0.shape[0] // y.shape[0])          # smoothing may stack eps_bsz copies
        return F.log_softmax(logits, dim=-1).gather(1, yy[:, None]).squeeze(1)
    return log_p


tfg_cfg = TFGConfig(rho=cfg.rho, mu=cfg.mu, sigma=cfg.sigma, eps_bsz=cfg.eps_bsz,
                    iter_steps=cfg.iter_steps, recur_steps=cfg.recur_steps,
                    clip_x0=CLIP_X0, eta=cfg.eta)

g = torch.Generator(device=DEV).manual_seed(0)
x0_probe = torch.randn(8, *meta["shape"], device=DEV, generator=g)
y_probe = torch.arange(8, device=DEV)

# check 1: guidance off == plain deterministic DDIM, same starting noise
off = TFG(eps_uncond, make_log_p(y_probe), ABAR, TFGConfig(rho=0, mu=0, iter_steps=0, clip_x0=CLIP_X0, eta=0.0))
a = off.sample(8, meta["shape"], 10, x_init=x0_probe)
b = ddim_sample(eps_uncond, 8, meta["shape"], 10, x_init=x0_probe, eta=0.0)
assert torch.allclose(a, b, atol=1e-4), (a - b).abs().max().item()
assert off.stats.objective_evals == 0 and off.stats.denoiser_backward == 0
print("rho = mu = 0 reproduces plain DDIM (eta = 0) exactly")

# check 2: guidance off == the notebook's own sampler at its own eta with identically seeded
# generators, so the sweep's scale-0 point is the uncond row, not a different sample.
off = TFG(eps_uncond, make_log_p(y_probe), ABAR, TFGConfig(rho=0, mu=0, iter_steps=0, clip_x0=CLIP_X0, eta=cfg.eta))
a = off.sample(8, meta["shape"], 10, x_init=x0_probe, generator=torch.Generator(device=DEV).manual_seed(5))
b = ddim_sample(eps_uncond, 8, meta["shape"], 10, x_init=x0_probe, generator=torch.Generator(device=DEV).manual_seed(5))
assert torch.allclose(a, b, atol=1e-4), (a - b).abs().max().item()
print(f"rho = mu = 0 reproduces the plain sampler at eta = {cfg.eta:g} with the same generator exactly")
print("TFG config:", {k: (v if not isinstance(v, tuple) else X0_CLAMP_DESC) for k, v in asdict(tfg_cfg).items()})

## Sample: four rows from the same starting noise

| row | what the base is told | guidance |
|---|---|---|
| uncond | the cell line; drug null | none |
| label | the cell line and the drug | none; the plain conditional reference (classifier-free guidance at w = 1) |
| TFG | the cell line; drug null | classifier gradient toward the pair's drug |
| TFG, wrong drug | the cell line; drug null | classifier gradient toward the next pair's drug |

The evaluation pairs are the largest held-out pair of each of the six most populous drugs, one
pair per drug (the six largest pairs outright are five cell lines of one drug, which would make
the wrong drug the same drug). `samples_per_condition` samples per pair; every row starts from
the same noise with identically seeded generators, so the only difference between rows is the
guidance. The wrong-drug row is the diagnostic: if guidance works, those samples should score as
the wrong drug. Each row is timed, and TFG's cost is reported as denoiser forward and backward
passes plus objective evaluations per sample, since it is not a single NFE.

The cell ends with a scale check: the total variance of each row over that of the real
validation cells. A row far above 1 has left the data range and nothing computed on it later
means anything.

In [ ]:
# Evaluation pairs: the largest held-out pair of each of the n_eval_conditions most populous
# drugs, PER samples each; every metric is per pair. One pair per drug so the wrong target is a
# different drug (the six largest pairs outright are five lines of one drug). Ties in count are
# broken by (line, drug) id so every cell notebook picks the same pairs.
counts = torch.bincount(val_loader.dataset.cond, minlength=meta["n_conditions"])
counts[meta["cond_ids_table"][:, 0] == 0] = 0                 # drop DMSO pairs
EVAL_CONDS, _seen = [], set()
for c in sorted(range(meta["n_conditions"]), key=lambda c: (-int(counts[c]), int(meta["cond_ids_table"][c, 1]), int(meta["cond_ids_table"][c, 0]))):
    d = int(meta["cond_ids_table"][c, 0])
    if counts[c] > 0 and d not in _seen:
        EVAL_CONDS.append(c); _seen.add(d)
    if len(EVAL_CONDS) == cfg.n_eval_conditions:
        break
assert len(EVAL_CONDS) >= 2, "need at least two drugs for a wrong-target row"
PER = cfg.samples_per_condition
y_eval = torch.tensor(EVAL_CONDS, device=DEV).repeat_interleave(PER)      # pair ids
DRUG_Y, LINE_Y = COND_TABLE[y_eval, 0], COND_TABLE[y_eval, 1]              # each sample's target drug and its line
DRUG_WRONG = DRUG_Y.roll(PER)                                             # the next pair's drug: a different drug by construction
assert bool((DRUG_WRONG != DRUG_Y).all()), "the wrong target must be a different drug for every sample"
N = len(y_eval)
NULL_DRUG = torch.full_like(DRUG_Y, meta["n_drugs"])
print("pairs (real validation cells):")
for c in EVAL_CONDS:
    print(f"   {meta['cond_text'][c]}   n={int(counts[c])}")

def eps_line(x, t):
    '''The base told the cell line but not the drug: TFG's "unconditional" model here, so that the
    classifier gradient supplies only what it can, the drug.'''
    return base_frozen.forward_ids(x, t, NULL_DRUG, LINE_Y)
g = torch.Generator(device=DEV).manual_seed(cfg.seed + 1)
X0 = torch.randn(N, *meta["shape"], device=DEV, generator=g)

def paired_gen():
    '''Fresh generator, same seed: identical internal randomness for every matched call.'''
    return torch.Generator(device=DEV).manual_seed(cfg.seed + 1234)

def timed(fn):
    if DEV.type == "cuda": torch.cuda.synchronize()
    t0 = time.time(); out = fn()
    if DEV.type == "cuda": torch.cuda.synchronize()
    return out, (time.time() - t0) / N

rows = {}
rows["uncond"], t_uncond = timed(lambda: ddim_sample(eps_line, N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen()))
rows["label"], t_label = timed(lambda: ddim_sample(lambda x, t: base_frozen(x, t, y_eval), N, meta["shape"],
                                                   cfg.sample_steps, x_init=X0, generator=paired_gen()))
tfg = TFG(eps_line, make_log_p(DRUG_Y), ABAR, tfg_cfg)    # Algorithm 1, their CIFAR-10 settings
rows["tfg"], t_tfg = timed(lambda: tfg.sample(N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen()))
tfg_stats = tfg.stats
tfg_w = TFG(eps_line, make_log_p(DRUG_WRONG), ABAR, tfg_cfg)
rows["tfg_wrong"], _ = timed(lambda: tfg_w.sample(N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen()))

# Fraction of coordinates on the clamp; the objective's gradient is zero there.
SATURATION = {k: saturation(v) for k, v in rows.items()}

print(f"\n{'row':>10} {'s / sample':>11}   denoiser fwd / bwd   objective calls   on the clamp")
print(f"{'uncond':>10} {t_uncond:>11.3f}   {cfg.sample_steps:>4} / {0:<4}          0          {SATURATION['uncond']:6.1%}")
print(f"{'label':>10} {t_label:>11.3f}   {cfg.sample_steps:>4} / {0:<4}          0          {SATURATION['label']:6.1%}")
print(f"{'tfg':>10} {t_tfg:>11.3f}   {tfg_stats.denoiser_evals:>4} / {tfg_stats.denoiser_backward:<4}          "
      f"{tfg_stats.objective_evals}        {SATURATION['tfg']:6.1%}")
print(f"{'tfg_wrong':>10} {'':>11}   {tfg_w.stats.denoiser_evals:>4} / {tfg_w.stats.denoiser_backward:<4}          "
      f"{tfg_w.stats.objective_evals}        {SATURATION['tfg_wrong']:6.1%}")

# ---- Scale check before any figure or metric: total variance of each row over that of the real
# validation cells. Far above 1 means the row left the data range and nothing computed on it
# means anything.
REAL_VAR = val_loader.dataset.x.var(0).sum().item()
SCALE = {k: v.var(0).sum().item() / REAL_VAR for k, v in rows.items()}
OUT_OF_RANGE = [k for k, s in SCALE.items() if not math.isfinite(s) or s > 5]
print("\nvariance ratio vs all real validation cells:", {k: round(s, 2) for k, s in SCALE.items()})
if OUT_OF_RANGE:
    print(f"\n*** SAMPLES OUT OF RANGE: {OUT_OF_RANGE}. Every number below for these rows is meaningless. "
          "If 'uncond' is flagged the base or the sampler is wrong; if only the TFG rows are, guidance "
          "diverged: lower rho and mu, and check the clamp. ***\n")

## Condition accuracy under the kNN scorer

The k-nearest-neighbour vote never touched the sampler. For each row: how often the scorer names
the drug that was asked for. For the wrong-drug row the number is against the drug that was
pushed toward, so a high value means guidance followed its target. Two ceilings are printed: the
scorer's accuracy on real validation cells overall, and on the real cells of these six pairs,
which is the one that applies (they are unseen pairs, so the reference set holds no cell of
them). A generated row can score above the real cells; that is over-steering, not success, which
is why accuracy is read together with the distribution metrics below.

In [ ]:
@torch.no_grad()
def acc_of(x, y):
    return (knn_label(x) == y).float().mean().item()

# The ceiling for these pairs: the scorer on their real validation cells. They are unseen pairs,
# so the reference set holds no cell of them and this is usually far below the all-pairs figure.
_real = [X_VAL[val_loader.dataset.cond.to(DEV) == c][:PER] for c in EVAL_CONDS]
REAL_PAIR_ACC = acc_of(torch.cat(_real), torch.cat([COND_TABLE[c, 0].repeat(len(r)) for c, r in zip(EVAL_CONDS, _real)]))

ACC = {
    "real_val_cells_knn": KNN_REAL_ACC,
    "real_val_cells_gaussian": LDA_REAL_ACC,
    "real_cells_of_eval_pairs_knn": REAL_PAIR_ACC,
    "uncond": acc_of(rows["uncond"], DRUG_Y),
    "label": acc_of(rows["label"], DRUG_Y),
    "tfg": acc_of(rows["tfg"], DRUG_Y),
    "tfg_wrong_vs_target": acc_of(rows["tfg_wrong"], DRUG_WRONG),
    "tfg_wrong_vs_original": acc_of(rows["tfg_wrong"], DRUG_Y),
}
for k, v in ACC.items():
    print(f"{k:>28}: {v:6.1%}")
print()
print(f"chance = {1 / N_DRUGS:.1%}.")
print("If guidance follows its target, 'tfg' sits above 'uncond' and 'tfg_wrong_vs_target' is comparably high.")
if OUT_OF_RANGE:
    print(f"rows {OUT_OF_RANGE} left the data range (scale check above): their accuracies are not evidence about steering.")
elif ACC["tfg"] < ACC["uncond"] + 0.02:
    print("little steering at these settings: try larger rho/mu or more iter_steps.")
print()
print("Accuracy alone rewards collapsed or over-steered samples even with a separate scorer: a row can score above "
      "the real cells of its own pair. Read it with the per-pair MMD, precision and variance ratio below.")

## Figure: first two principal components

Real validation cells of the six pairs, then each generated row, one colour per pair. The axes
are fixed by the real cells and shared by all panels; a row with points outside them gets that
share written in its title.

In [ ]:
import matplotlib.pyplot as plt

real_x = val_loader.dataset.x.numpy(); real_c = val_loader.dataset.cond.numpy()
gen_c = y_eval.cpu().numpy()
cmap = plt.get_cmap("tab10")

# Axes fixed by the real cells of the evaluation pairs (1st to 99th percentile, padded) and shared
# by every panel, so an out-of-range row cannot flatten the others; its off-panel share goes in
# the title.
real_sel = np.concatenate([real_x[real_c == c][:PER] for c in EVAL_CONDS])
_lo, _hi = np.percentile(real_sel[:, :2], 1, axis=0), np.percentile(real_sel[:, :2], 99, axis=0)
_pad = 0.5 * (_hi - _lo)
XLIM, YLIM = (_lo[0] - _pad[0], _hi[0] + _pad[0]), (_lo[1] - _pad[1], _hi[1] + _pad[1])

def pair_label(c):
    '''"drug / cell line", not the drug alone: several pairs can share a drug.'''
    t = meta["cond_text"][c]
    return t.split(",")[0] + " / " + t.split(" in ")[-1].replace(" cells", "")

fig, ax = plt.subplots(1, 5, figsize=(23, 4.4), sharex=True, sharey=True)
panels = [("real (val)", real_x, real_c), ("uncond: line only", rows["uncond"].cpu().numpy(), gen_c),
          ("label: line + drug", rows["label"].cpu().numpy(), gen_c),
          ("TFG toward the drug", rows["tfg"].cpu().numpy(), gen_c),
          ("TFG, wrong drug", rows["tfg_wrong"].cpu().numpy(), gen_c)]
for a, (name, pts, cs) in zip(ax, panels):
    n_off = n_pts = 0
    for j, c in enumerate(EVAL_CONDS):
        p = pts[cs == c][:PER]
        a.scatter(p[:, 0], p[:, 1], s=6, alpha=0.5, color=cmap(j), label=pair_label(c) if a is ax[0] else None)
        n_off += int(((p[:, 0] < XLIM[0]) | (p[:, 0] > XLIM[1]) | (p[:, 1] < YLIM[0]) | (p[:, 1] > YLIM[1])).sum())
        n_pts += len(p)
    a.set_title(name + (f"  ({n_off / max(n_pts, 1):.0%} off panel)" if n_off else ""))
    a.set_xlabel("PC1")
ax[0].set_xlim(*XLIM); ax[0].set_ylim(*YLIM)
ax[0].set_ylabel("PC2"); ax[0].legend(fontsize=7, markerscale=2); plt.tight_layout(); plt.show()

## Guidance-strength sweep

`rho` and `mu` scaled together by the factors in `cfg.sweep`. Scale 0 must reproduce the uncond
row exactly (asserted). Each point reports the kNN accuracy, the variance ratio and the on-clamp
fraction, so an out-of-range point is visible as such. This is a first pass, not a
hyperparameter search: pick a setting here, then report it with fresh sampling seeds; do not
report the maximum of the sweep.

In [ ]:
SWEEP = list(cfg.sweep)
sweep_rows = []
for scale in SWEEP:
    c = TFGConfig(rho=cfg.rho * scale, mu=cfg.mu * scale, sigma=cfg.sigma, eps_bsz=cfg.eps_bsz,
                  iter_steps=cfg.iter_steps if scale > 0 else 0, recur_steps=cfg.recur_steps,
                  clip_x0=CLIP_X0, eta=cfg.eta)
    s, dt = timed(lambda: TFG(eps_line, make_log_p(DRUG_Y), ABAR, c).sample(
        N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen()))
    if scale == 0:
        # Guidance off must be the uncond row: same starting noise, same generator, same clamp.
        assert torch.allclose(s, rows["uncond"], atol=1e-4), "scale 0 of the sweep differs from the uncond row"
    sweep_rows.append({"scale": scale, "rho": c.rho, "mu": c.mu, "acc": acc_of(s, DRUG_Y),
                       "var_ratio_vs_val": s.var(0).sum().item() / REAL_VAR, "on_clamp": saturation(s), "s_per_sample": dt})
    r = sweep_rows[-1]
    print(f"scale {scale:>4}  rho {c.rho:>5.2f}  mu {c.mu:>5.3f}  acc {r['acc']:6.1%}  var ratio {r['var_ratio_vs_val']:7.2f}"
          f"  on clamp {r['on_clamp']:5.1%}  {dt:.3f}s/sample" + ("   <-- out of range" if r["var_ratio_vs_val"] > 5 else ""))

fig, ax = plt.subplots(figsize=(5.5, 3.6))
ok = [r for r in sweep_rows if r["var_ratio_vs_val"] <= 5]
bad = [r for r in sweep_rows if r["var_ratio_vs_val"] > 5]
ax.plot([r["scale"] for r in sweep_rows], [r["acc"] for r in sweep_rows], marker="o", label="TFG (kNN accuracy)")
if bad:
    ax.scatter([r["scale"] for r in bad], [r["acc"] for r in bad], marker="x", s=80, c="red", zorder=5,
               label="samples out of range (variance ratio > 5)")
ax.axhline(ACC["label"], ls="--", c="grey", lw=1, label=f"line + drug conditioned base ({ACC['label']:.1%})")
ax.axhline(ACC["real_cells_of_eval_pairs_knn"], ls="-.", c="grey", lw=1,
           label=f"real cells of these pairs ({ACC['real_cells_of_eval_pairs_knn']:.1%})")
ax.axhline(1 / N_DRUGS, ls=":", c="grey", lw=1, label="chance")
ax.set_xlabel("guidance scale (x default rho, mu)"); ax.set_ylabel(f"accuracy ({KNN_K}-NN scorer)")
ax.legend(fontsize=7); plt.tight_layout(); plt.show()

## Per-pair metrics

From `src/guidance/eval/cell_metrics.py`, per pair against its real validation cells: MMD, energy
distance and Fréchet distance on the PCs (distance between the two populations), precision and
recall (quality and coverage), the variance ratio (far below 1 is collapse), and drug accuracy
under a k-nearest-neighbour scorer over real training cells that never touched the sampler. The
first block of rows is the real cells themselves; their kNN accuracy is the ceiling that applies
to these unseen pairs, and a generated row that scores above it is over-steered or collapsed, not
better. The wrong-drug row is scored against the pair that was requested.

In [ ]:
# src/guidance/eval/cell_metrics.py, per pair against its real validation cells: MMD, energy
# distance, Frechet distance on the PCs, precision, recall, variance ratio, kNN drug accuracy.
# The wrong-drug row is scored against the pair that was requested.
val_x, val_c = val_loader.dataset.x, val_loader.dataset.cond
gen_c = y_eval.cpu()
METRICS = {"eval_conditions": EVAL_CONDS, "samples_per_condition": PER,
           "scorer": SCORER.describe(), "scorer_real_val_drug_accuracy": KNN_REAL_ACC}
print(f"{'pair':30s} {'arm':10s} {'MMD':>6} {'energy':>7} {'FD_pc':>7} {'prec':>5} {'rec':>5} {'var':>6} {'acc':>6} {'n real':>6}")
# Reference rows: the scorer on each pair's own real validation cells. A generated row that scores
# above it is over-steered or collapsed, not better.
real_rows = []
for c in EVAL_CONDS:
    r = val_x[val_c == c]
    acc = SCORER.accuracy(r[:PER], COND_TABLE[c, 0].repeat(min(PER, len(r))))
    real_rows.append({"cond": int(c), "text": meta["cond_text"][c], "cond_acc": acc, "n_real": int(len(r))})
    print(f"{meta['cond_text'][c][:30]:30s} {'real (val)':10s} {'-':>6} {'-':>7} {'-':>7} {'-':>5} {'-':>5} {'1.00':>6} {acc:6.1%} {len(r):6d}")
METRICS["real_per_condition"] = real_rows
METRICS["real_cond_acc_mean"] = float(np.mean([row["cond_acc"] for row in real_rows]))
for name in ("uncond", "label", "tfg", "tfg_wrong"):
    per = []
    for c in EVAL_CONDS:
        r = val_x[val_c == c]; gsel = rows[name][gen_c == c]
        m = all_metrics(gsel, r, seed=cfg.seed)
        m["cond_acc"] = SCORER.accuracy(gsel, COND_TABLE[c, 0].repeat(len(gsel)))
        per.append({"cond": int(c), "text": meta["cond_text"][c], **m})
        print(f"{meta['cond_text'][c][:30]:30s} {name:10s} {m['mmd']:6.3f} {m['energy']:7.3f} {m['frechet_pc']:7.2f} "
              f"{m['precision']:5.2f} {m['recall']:5.2f} {m['var_ratio']:6.2f} {m['cond_acc']:6.1%} {len(r):6d}")
    METRICS[f"{name}_per_condition"] = per
    for k in ("mmd", "energy", "frechet_pc", "precision", "recall", "var_ratio", "cond_acc"):
        METRICS[f"{name}_{k}_mean"] = float(np.mean([p_[k] for p_ in per]))
if OUT_OF_RANGE:
    print(f"\n*** rows {OUT_OF_RANGE} left the data range (scale check above): their metrics are not results. ***")
FID_SCORES = {}

## Manifest

Everything reported above, plus the commit, config, hardware, base source and clamp setting,
written to `manifest_tfg_cells.json`.

In [ ]:
report = {
    "method": "TFG training-free guidance (arXiv:2409.15761), per-drug Gaussian classifier on cell states",
    "modality": "cells",
    "code_version": GIT_SHA,
    "config": asdict(cfg),
    "tfg_config": {k: (X0_CLAMP_DESC if k == "clip_x0" else v) for k, v in asdict(tfg_cfg).items()},
    "x0_clamp": X0_CLAMP_DESC,
    "base_source": BASE_SOURCE,
    "unconditional_branch": "cell line given, drug null (eps_line)",
    "guide_classifier": f"Gaussian classifier (LDA), {N_DRUGS} drugs, shared covariance, shrinkage {cfg.shrinkage}, fitted on {len(X_TRAIN):,} training cells",
    "eval_classifier": f"{KNN_K}-nearest-neighbour vote over {len(KNN_REF):,} real training cells with true drug labels",
    "label_path": False,
    "accuracy": ACC,
    "n_eval_samples": N,
    "sample_variance_ratio_vs_val": SCALE,
    "samples_out_of_range": OUT_OF_RANGE,
    "on_clamp_fraction": SATURATION,
    "sweep": sweep_rows,
    "seconds_per_sample": {"uncond": t_uncond, "label": t_label, "tfg": t_tfg},
    "tfg_cost_per_sample": {"denoiser_forward": tfg_stats.denoiser_evals,
                            "denoiser_backward": tfg_stats.denoiser_backward,
                            "objective_evals": tfg_stats.objective_evals},
    "plain_ddim_cost_per_sample": {"denoiser_forward": cfg.sample_steps},
    "base_best_val": min(h["val"] for h in base_hist),
    "checkpoint_meta": CKPT_META,
    "trainable_params": 0,
    "frozen_backbone": sum(p.numel() for p in base_frozen.parameters()),
    "nfe_sample_steps": cfg.sample_steps,
    "train_seconds": {"base": sum(h.get("seconds", 0) for h in base_hist)},
    "hardware": torch.cuda.get_device_name(0) if DEV.type == "cuda" else "cpu",
    "device": str(DEV),
}
report.update(globals().get("FID_SCORES", {}))
report.update(globals().get("METRICS", {}))
print(json.dumps(report, indent=2))
with open("manifest_tfg_cells.json", "w") as f:
    json.dump(report, f, indent=2)

## Before reporting any of this

- The scale check flagged nothing (`samples_out_of_range` is empty in the manifest).
- Both scorers' accuracy on real cells, and the kNN accuracy on the real cells of the evaluated
  pairs, are shown next to every generated number.
- `tfg_wrong_vs_target` is comparably high to `tfg`, so guidance follows its target.
- MMD, precision and the variance ratio are read together with accuracy, never accuracy alone.
- Three sampling seeds on one checkpoint, mean and standard deviation, stated as such.
- The guidance setting was chosen on the sweep and then reported with fresh seeds.
- Time per sample and the denoiser / objective call counts are shown next to the accuracy.
- The clamp (per-PC training range) and the on-clamp fraction are stated.
- `condition`, `dose`, `shrinkage` and `base_source` are stated.
- Described as TFG re-expressed against our sampler: their CIFAR-10 hyperparameters as starting
  settings, ancestral sampling (`eta = 1`, 250 steps), a Gaussian classifier fitted by moments
  in place of a pretrained network, and a conditional model with the line given and the drug
  null standing in for an unconditional one.